# 03E — Build remotely and run a managed training pipeline

Edit model code → ZIP the build inputs → CodeBuild builds and pushes to ECR →
publish a development pipeline → run Prepare, Train and Evaluate.

Run this notebook from `examples/notebooks/` inside the application checkout.
It uses **boto3 and the existing source files**; the notebook does not require
Docker, the SageMaker Python SDK or CDK. Docker runs on the remote builder.
The AWS calls are visible below; an error stops its cell with the original exception.

**Provide once:** an approved S3 bucket/prefix, a build-only CodeBuild project,
the SageMaker pipeline role, and a separate SageMaker job role.
Use a namespace and repository those roles are permitted to access.
The build project must accept an S3 ZIP with `buildspec.yml`, run Linux x86 Docker
with BuildKit, and have access to the base image, package sources and destination ECR
repository. Its service role needs source-S3 read, ECR pull/push and build-log access.
It does not need CDK, CloudFormation or deployment-role permissions.

The notebook caller needs CodeBuild start/inspect, ECR describe (create/tag only for
a new repository), S3 input/output access, SageMaker create/update/start/inspect,
`iam:PassRole` for the pipeline role, and log reads. The pipeline role passes the
job role and orchestrates jobs; the job role reads the model image and input data
and writes outputs. KMS and VPC permissions depend on the supplied platform settings.

If no approved builder exists yet, the platform team must provision that project
and service role first. ECR stores images; it does not build them.
This development notebook does not change IAM or the build project's configuration.


## 1. Enter the platform bindings

- Set the four required platform values: S3 prefix, CodeBuild project and two roles.
- Choose a model and namespace; leave the base override empty to use its checked-in default.
- Add the approved base or network/key settings here when your environment requires them.


In [ ]:
MODEL = "usecase2"
NAMESPACE = "samplemlops"
DEVELOPMENT = "notebook"

S3_ROOT = ""                 # s3://approved-bucket/approved-prefix
CODEBUILD_PROJECT = ""       # Existing, approved build-only project
PIPELINE_ROLE_ARN = ""       # SageMaker pipeline orchestration role
JOB_ROLE_ARN = ""            # Separate SageMaker Processing/Training role

REPOSITORY_NAME = f"{NAMESPACE}/{MODEL}"  # Or an existing approved repository
INPUT_DATA_URI = ""          # Empty uploads this model's small example CSV
BASE_IMAGE = ""              # Empty uses the model default; overrides require @sha256
OS_SETUP = "debian"          # Use "provided" when the base owner maintains OS packages
INSTANCE_TYPE = "ml.m5.large"

# Leave empty only when the platform permits the default network and encryption.
INFRASTRUCTURE = {}
# INFRASTRUCTURE = {
#     "network": {"subnet_ids": ["subnet-..."], "security_group_ids": ["sg-..."],
#                 "isolation": True},
#     "output_kms_key_arn": "arn:aws:kms:REGION:ACCOUNT:key/KEY",
#     "volume_kms_key_arn": "arn:aws:kms:REGION:ACCOUNT:key/KEY",
# }
S3_KMS_KEY_ARN = ""          # Explicit key for notebook uploads, if required


## 2. Confirm identity and local source

- Use the notebook's normal AWS session and show the actual account and Region.
- Locate the application and import its existing build-context and definition functions.
- Check the required settings and print the development pipeline and output prefix.


In [ ]:
import hashlib
import json
from pathlib import Path
import sys
import time
from urllib.parse import urlparse
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import boto3

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "mlops-models/src/mlops_models").is_dir())
sys.path.insert(0, str(ROOT / "mlops-models"))
sys.path.insert(0, str(ROOT / "mlops-models/src"))
from containers.build import prepare_context, build_arguments, context_fingerprint
from mlops_skeleton.pipelines import load_definition
from mlops_skeleton.names import validate_namespace, validate_model_id

session = boto3.Session()
identity = session.client("sts").get_caller_identity()
ACCOUNT, REGION = identity["Account"], session.region_name
s3 = session.client("s3")
ecr = session.client("ecr")
codebuild = session.client("codebuild")
sagemaker = session.client("sagemaker")
logs = session.client("logs")
print(json.dumps({"account": ACCOUNT, "region": REGION, "caller": identity["Arn"]}, indent=2))

assert REGION, "Set the AWS Region in your notebook environment."
assert all([S3_ROOT, CODEBUILD_PROJECT, PIPELINE_ROLE_ARN, JOB_ROLE_ARN]), "Fill the four platform bindings in cell 1."
assert PIPELINE_ROLE_ARN != JOB_ROLE_ARN, "Use separate pipeline and container job roles."
validate_namespace(NAMESPACE)
validate_model_id(MODEL)
storage = urlparse(S3_ROOT)
BUCKET, PREFIX = storage.netloc, storage.path.strip("/")
assert storage.scheme == "s3" and BUCKET and PREFIX, "S3_ROOT must include a bucket and a nonempty prefix."
S3_ROOT = f"s3://{BUCKET}/{PREFIX}"
PIPELINE_NAME = f"{NAMESPACE}-{MODEL}-training-dev-{DEVELOPMENT}"
OUTPUT_URI = f"{S3_ROOT}/models/{MODEL}/runs"
print("Pipeline:", PIPELINE_NAME)
print("Run outputs:", OUTPUT_URI)


## 3. Inspect the remote builder

- Read the selected project and display its worker image, service role and network.
- Require the approved project's normal `buildspec.yml` and S3-source contract.
- Leave its permissions, worker image and Docker configuration unchanged.


In [ ]:
project_result = codebuild.batch_get_projects(names=[CODEBUILD_PROJECT])
assert project_result["projects"], f"CodeBuild project not found: {CODEBUILD_PROJECT}"
project = project_result["projects"][0]
print(json.dumps({
    "project": project["name"], "service_role": project["serviceRole"],
    "source": project["source"], "worker_image": project["environment"]["image"],
    "worker_type": project["environment"]["type"],
    "privileged_mode": project["environment"].get("privilegedMode", False),
    "vpc": project.get("vpcConfig", {}),
}, indent=2))
assert project["source"]["type"] == "S3", "Use an approved build-only project configured for S3 source."
assert project["source"].get("buildspec", "") in ("", "buildspec.yml"), "This project uses a different build contract; ask its owner before changing it."


## 4. Reuse or create the image repository

- Look up the repository in the current account and Region.
- If it does not exist, create it with immutable tags and scan-on-push.
- Print its URI; access errors are not treated as a missing repository.


In [ ]:
try:
    repository = ecr.describe_repositories(repositoryNames=[REPOSITORY_NAME])["repositories"][0]
except ecr.exceptions.RepositoryNotFoundException:
    repository = ecr.create_repository(
        repositoryName=REPOSITORY_NAME,
        imageTagMutability="IMMUTABLE",
        imageScanningConfiguration={"scanOnPush": True},
        tags=[{"Key": "Deployment", "Value": NAMESPACE}],
    )["repository"]
REPOSITORY_URI = repository["repositoryUri"]
print("Repository:", REPOSITORY_URI)
print("Tag policy:", repository["imageTagMutability"])


## 5. Prepare this model's build inputs

- Copy the shared runtime, selected model, existing Dockerfile and hash-locked dependencies.
- Apply the approved base override when supplied, keeping model code unchanged.
- Create a new development image tag; only local files are written in this cell.


In [ ]:
container_overrides = {"os_setup": OS_SETUP, "platform": "linux/amd64"}
if BASE_IMAGE:
    container_overrides["base_image"] = BASE_IMAGE
context, image_settings = prepare_context(MODEL, {"container": container_overrides})
context_sha256 = context_fingerprint(context, image_settings)
build_id = uuid4().hex[:12]
IMAGE_TAG = f"dev-{context_sha256[:12]}-{build_id}"
IMAGE_URI = f"{REPOSITORY_URI}:{IMAGE_TAG}"
WORK = ROOT / ".runs/remote-build" / build_id
WORK.mkdir(parents=True, exist_ok=True)
print("Context:", context)
print("Model folders:", [p.name for p in (context / "mlops_models/models").iterdir() if p.is_dir()])
print("Base:", image_settings["base_image"])
print("New image:", IMAGE_URI)


## 6. Write the commands CodeBuild will execute

- Authenticate the builder to the destination ECR registry and, when needed, the private base registry.
- Build using the same arguments and Docker manifest format as the existing local build.
- Push the image only after a successful build; credentials are not included in the source ZIP.

The JSON below is valid YAML and is saved as `buildspec.yml`.
The build project's service role supplies AWS credentials inside CodeBuild.


In [ ]:
base_registry = image_settings["base_image"].split("/", 1)[0]
build_environment = {
    **build_arguments(MODEL, image_settings),
    "IMAGE_URI": IMAGE_URI,
    "IMAGE_PLATFORM": image_settings["platform"],
    "TARGET_REGISTRY": REPOSITORY_URI.split("/", 1)[0],
    "TARGET_REGION": REGION,
    "DOCKER_BUILDKIT": "1",
}
commands = [
    "set -euo pipefail",
    'aws ecr get-login-password --region "$TARGET_REGION" | docker login --username AWS --password-stdin "$TARGET_REGISTRY"',
]
if base_registry != "public.ecr.aws":
    build_environment.update(BASE_REGISTRY=base_registry, BASE_REGION=base_registry.split(".")[3])
    commands.append('aws ecr get-login-password --region "$BASE_REGION" | docker login --username AWS --password-stdin "$BASE_REGISTRY"')
commands += [
    'docker build --platform "$IMAGE_PLATFORM" --provenance=false --output type=docker,oci-mediatypes=false --build-arg BASE_IMAGE="$BASE_IMAGE" --build-arg MODEL_ID="$MODEL_ID" --build-arg OS_SETUP="$OS_SETUP" --tag "$IMAGE_URI" .',
    'docker push "$IMAGE_URI"',
]
buildspec = {
    "version": 0.2,
    "env": {"shell": "bash", "variables": build_environment},
    "phases": {"build": {"commands": commands}},
}
(context / "buildspec.yml").write_text(json.dumps(buildspec, indent=2) + "\n")
print((context / "buildspec.yml").read_text())


## 7. Upload the source ZIP

- Archive the contents of the build directory, with Dockerfile and buildspec at the ZIP root.
- Upload under the supplied S3 prefix and record the object version when available.
- Print the exact source URI and checksum; this sends files, not a locally built image.


In [ ]:
source_zip = WORK / "source.zip"
with ZipFile(source_zip, "w", ZIP_DEFLATED) as archive:
    for path in sorted(context.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(context).as_posix())
source_sha256 = hashlib.sha256(source_zip.read_bytes()).hexdigest()
source_key = f"{PREFIX}/models/{MODEL}/builds/{build_id}/source.zip"
upload_encryption = {}
if S3_KMS_KEY_ARN:
    upload_encryption = {"ServerSideEncryption": "aws:kms", "SSEKMSKeyId": S3_KMS_KEY_ARN}
with source_zip.open("rb") as body:
    source_object = s3.put_object(
        Bucket=BUCKET, Key=source_key, Body=body, ContentType="application/zip",
        Metadata={"sha256": source_sha256}, IfNoneMatch="*", **upload_encryption,
    )
print("Source:", f"s3://{BUCKET}/{source_key}")
print("Version:", source_object.get("VersionId", "bucket is not versioned"))
print("SHA-256:", source_sha256)


## 8. Start the image build

- Submit this source ZIP to the selected build-only project.
- Pin the S3 object version when the bucket provides one.
- Print the build ID; no project, role, privileged-mode or buildspec override is sent.


In [ ]:
source_version = source_object.get("VersionId")
version_argument = {"sourceVersion": source_version} if source_version and source_version != "null" else {}
build = codebuild.start_build(
    projectName=CODEBUILD_PROJECT,
    sourceLocationOverride=f"{BUCKET}/{source_key}",
    idempotencyToken=build_id,
    **version_argument,
)["build"]
BUILD_ID = build["id"]
(WORK / "build-start.json").write_text(json.dumps(build, indent=2, default=str) + "\n")
print("Build:", BUILD_ID)


## 9. Wait for the build

- Poll CodeBuild and print each phase as it changes.
- Save the final build response and print failure details if present.
- A failed build blocks image publication; inspect its logs in the next cell.


In [ ]:
deadline = time.monotonic() + 3600
previous_phase = None
while True:
    build = codebuild.batch_get_builds(ids=[BUILD_ID])["builds"][0]
    phase = (build["currentPhase"], build["buildStatus"])
    if phase != previous_phase:
        print(*phase, flush=True)
        previous_phase = phase
    if build["buildComplete"]:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError(f"Build still running: {BUILD_ID}. Rerun this cell to continue waiting.")
    time.sleep(15)
(WORK / "build.json").write_text(json.dumps(build, indent=2, default=str) + "\n")
for phase in build.get("phases", []):
    if phase.get("phaseStatus") in {"FAILED", "FAULT", "TIMED_OUT"}:
        print(json.dumps(phase, indent=2, default=str))
print("Logs:", build.get("logs", {}).get("deepLink", "See CodeBuild → build → Logs"))
assert build["buildStatus"] == "SUCCEEDED", f"Build {BUILD_ID} failed; inspect the next cell."


## 10. Read the build log

- Retrieve the selected build's CloudWatch log stream, including all pages.
- Save the full text locally and print its last 80 lines.
- Run this cell after either a successful or failed build.


In [ ]:
build = codebuild.batch_get_builds(ids=[BUILD_ID])["builds"][0]
build_log = build.get("logs", {})
log_lines, token = [], None
if build_log.get("groupName") and build_log.get("streamName"):
    while True:
        page = logs.get_log_events(
            logGroupName=build_log["groupName"], logStreamName=build_log["streamName"],
            startFromHead=True, **({"nextToken": token} if token else {}),
        )
        log_lines.extend(item["message"] for item in page["events"])
        if page["nextForwardToken"] == token:
            break
        token = page["nextForwardToken"]
    (WORK / "build.log").write_text("\n".join(log_lines) + "\n")
    print("\n".join(log_lines[-80:]))
    print("Full log:", WORK / "build.log")
else:
    print("No CloudWatch stream was returned. Inspect CodeBuild phases above.")


## 11. Resolve the pushed image to its digest

- Require a successful build before looking up its unique image tag.
- Read the actual ECR manifest and check its Docker V2 format for SageMaker compatibility.
- Use the immutable digest URI for all pipeline steps.


In [ ]:
assert build["buildStatus"] == "SUCCEEDED", "Fix the image build before continuing."
image_response = ecr.batch_get_image(
    repositoryName=REPOSITORY_NAME,
    imageIds=[{"imageTag": IMAGE_TAG}],
)
assert not image_response.get("failures"), image_response.get("failures")
image = image_response["images"][0]
assert image["imageManifestMediaType"] == "application/vnd.docker.distribution.manifest.v2+json", "Rebuild using the Docker V2 exporter shown above."
IMAGE_DIGEST = image["imageId"]["imageDigest"]
RUNTIME_IMAGE = f"{REPOSITORY_URI}@{IMAGE_DIGEST}"
print("Runtime image:", RUNTIME_IMAGE)


## 12. Select or upload the training input

- Use the supplied S3 input URI if one was entered.
- Otherwise upload the selected model's checked-in synthetic CSV under this build's prefix.
- Print the input URI that SageMaker will copy into Prepare.


In [ ]:
model_config = json.loads((ROOT / "mlops-models/src/mlops_models/models" / MODEL / "model-config.json").read_text())
if not INPUT_DATA_URI:
    example = ROOT / "examples/data" / model_config["example_input"]
    input_key = f"{PREFIX}/models/{MODEL}/builds/{build_id}/{example.name}"
    s3.put_object(
        Bucket=BUCKET, Key=input_key, Body=example.read_bytes(),
        ContentType="text/csv", **upload_encryption,
    )
    INPUT_DATA_URI = f"s3://{BUCKET}/{input_key}"
print("Training input:", INPUT_DATA_URI)


## 13. Render the existing pipeline definition

- Load this model's native training definition, falling back to model_base.
- Substitute the built image, job role, S3 paths and platform network/key settings.
- Save and print the exact definition that the next cell sends to SageMaker.

No second pipeline recipe is maintained in this notebook. Prepare includes the
model's data preparation and feature selection; Train and Evaluate use the same
entrypoints as the Python and Local notebooks. Registration remains off for this
development run. Its referenced registry group is needed only if registration is enabled.


In [ ]:
definition = load_definition(
    MODEL, "training",
    namespace=NAMESPACE, pipeline_name=PIPELINE_NAME,
    role=JOB_ROLE_ARN, image=RUNTIME_IMAGE, image_digest=IMAGE_DIGEST,
    model_package_group=f"{NAMESPACE}-{MODEL}",
    input_uri=INPUT_DATA_URI, output_uri=OUTPUT_URI,
    infrastructure=INFRASTRUCTURE,
)
definition["Metadata"]["BuildSource"] = f"s3://{BUCKET}/{source_key}"
definition["Metadata"]["BuildSourceSHA256"] = source_sha256
(WORK / "definition.json").write_text(json.dumps(definition, indent=2) + "\n")
print(json.dumps(definition, indent=2))


## 14. Create or update the development pipeline

- Create the named pipeline if it does not exist, passing the pipeline orchestration role.
- Update an existing pipeline only when its deployment/development ownership tags match.
- Print the ARN; the pipeline is now visible in Studio and can also be run from the UI.


In [ ]:
try:
    existing_pipeline = sagemaker.describe_pipeline(PipelineName=PIPELINE_NAME)
except sagemaker.exceptions.ResourceNotFound:
    pipeline = sagemaker.create_pipeline(
        PipelineName=PIPELINE_NAME,
        RoleArn=PIPELINE_ROLE_ARN,
        PipelineDefinition=json.dumps(definition),
        Tags=[{"Key": "Deployment", "Value": NAMESPACE},
              {"Key": "Development", "Value": DEVELOPMENT}],
    )
else:
    tags = {tag["Key"]: tag["Value"] for page in sagemaker.get_paginator("list_tags").paginate(
        ResourceArn=existing_pipeline["PipelineArn"]) for tag in page["Tags"]}
    assert tags.get("Deployment") == NAMESPACE and tags.get("Development") == DEVELOPMENT, "Existing pipeline belongs to a different deployment; choose another development name."
    pipeline = sagemaker.update_pipeline(
        PipelineName=PIPELINE_NAME,
        RoleArn=PIPELINE_ROLE_ARN,
        PipelineDefinition=json.dumps(definition),
    )
print("Pipeline:", pipeline["PipelineArn"])


## 15. Start a managed execution

- Start Prepare → Train → Evaluate and the quality/registration conditions.
- Pass the instance types explicitly and leave model registration disabled.
- Print the execution ARN and the folder where SageMaker writes this run's outputs.


In [ ]:
execution = sagemaker.start_pipeline_execution(
    PipelineName=PIPELINE_NAME,
    PipelineExecutionDisplayName=f"train-{build_id}",
    PipelineParameters=[
        {"Name": "PrepareInstanceType", "Value": INSTANCE_TYPE},
        {"Name": "TrainInstanceType", "Value": INSTANCE_TYPE},
        {"Name": "EvaluateInstanceType", "Value": INSTANCE_TYPE},
        {"Name": "RegisterModel", "Value": "false"},
    ],
)
EXECUTION_ARN = execution["PipelineExecutionArn"]
EXECUTION_ID = EXECUTION_ARN.rsplit("/", 1)[-1]
RUN_URI = f"{OUTPUT_URI}/{EXECUTION_ID}"
(WORK / "execution.json").write_text(json.dumps(execution, indent=2) + "\n")
print("Execution:", EXECUTION_ARN)
print("Run outputs:", RUN_URI)


## 16. Wait for the pipeline

- Poll the actual managed execution and print its changing status.
- Print the original failure reason if the execution fails.
- A notebook timeout does not stop AWS jobs; rerun this cell to continue observing.


In [ ]:
deadline = time.monotonic() + 3600
previous_status = None
while True:
    execution_detail = sagemaker.describe_pipeline_execution(PipelineExecutionArn=EXECUTION_ARN)
    status = execution_detail["PipelineExecutionStatus"]
    if status != previous_status:
        print(status, flush=True)
        previous_status = status
    if status not in {"Executing", "Stopping"}:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError(f"Execution still running: {EXECUTION_ARN}. Rerun this cell to continue waiting.")
    time.sleep(20)
(WORK / "execution-detail.json").write_text(json.dumps(execution_detail, indent=2, default=str) + "\n")
print(execution_detail.get("FailureReason", "Execution finished."))


## 17. Inspect every step and its job logs

- List the pipeline's steps, status, failures and underlying job ARNs.
- Read each Processing/Training job's CloudWatch logs within this execution's time window.
- Save full logs locally and display the last 20 lines per job; useful after failures too.


In [ ]:
steps = [step for page in sagemaker.get_paginator("list_pipeline_execution_steps").paginate(
    PipelineExecutionArn=EXECUTION_ARN) for step in page["PipelineExecutionSteps"]]
print(json.dumps(steps, indent=2, default=str))
for step in steps:
    metadata = step.get("Metadata", {})
    for kind, group in [("ProcessingJob", "/aws/sagemaker/ProcessingJobs"),
                        ("TrainingJob", "/aws/sagemaker/TrainingJobs")]:
        if kind not in metadata:
            continue
        job_name = metadata[kind]["Arn"].rsplit("/", 1)[-1]
        job_lines = []
        for page in logs.get_paginator("filter_log_events").paginate(
            logGroupName=group, logStreamNamePrefix=job_name + "/",
            startTime=int(execution_detail["CreationTime"].timestamp() * 1000),
            endTime=int(time.time() * 1000),
        ):
            job_lines.extend(event["message"] for event in page["events"])
        (WORK / f"{step['StepName']}.log").write_text("\n".join(job_lines) + "\n")
        print("\n" + step["StepName"], job_name)
        print("\n".join(job_lines[-20:]))


## 18. Inspect the saved outputs

- List all objects in this execution's folder, without downloading the training dataset.
- Read and display the evaluation report after a successful run.
- Print a compact record of source, build, image, pipeline and execution for follow-up.


In [ ]:
run_location = urlparse(RUN_URI)
run_prefix = run_location.path.strip("/") + "/"
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=run_location.netloc, Prefix=run_prefix):
    for obj in page.get("Contents", []):
        print(obj["Key"][len(run_prefix):], obj["Size"], "bytes")
assert execution_detail["PipelineExecutionStatus"] == "Succeeded", "Inspect the failed steps and logs above."
report_object = s3.get_object(
    Bucket=run_location.netloc, Key=run_prefix + "reports/evaluation/evaluation.json",
)
report = json.loads(report_object["Body"].read())
print(json.dumps(report, indent=2))
receipt = {
    "source": f"s3://{BUCKET}/{source_key}", "source_version": source_version,
    "source_sha256": source_sha256, "codebuild": BUILD_ID,
    "image": RUNTIME_IMAGE, "pipeline": PIPELINE_NAME,
    "execution": EXECUTION_ARN, "outputs": RUN_URI,
}
(WORK / "result.json").write_text(json.dumps(receipt, indent=2) + "\n")
print(json.dumps(receipt, indent=2))
print("Local inspection files:", WORK)


## 19. Remove the development pipeline when finished

- Leave `REMOVE_PIPELINE = False` to retain the pipeline for inspection and UI runs.
- When enabled, confirm its ownership and refuse deletion while any execution is active.
- Delete only the pipeline definition; image, source ZIP, job logs and run outputs remain.


In [ ]:
REMOVE_PIPELINE = False
if REMOVE_PIPELINE:
    current = sagemaker.describe_pipeline(PipelineName=PIPELINE_NAME)
    tags = {tag["Key"]: tag["Value"] for page in sagemaker.get_paginator("list_tags").paginate(
        ResourceArn=current["PipelineArn"]) for tag in page["Tags"]}
    assert tags.get("Deployment") == NAMESPACE and tags.get("Development") == DEVELOPMENT, "Ownership tags do not match."
    for page in sagemaker.get_paginator("list_pipeline_executions").paginate(PipelineName=PIPELINE_NAME):
        assert not any(run["PipelineExecutionStatus"] in {"Executing", "Stopping"}
                       for run in page["PipelineExecutionSummaries"]), "Wait for or stop active executions first."
    print(sagemaker.delete_pipeline(PipelineName=PIPELINE_NAME))
else:
    print("Retained:", PIPELINE_NAME)


To test another code edit, edit the Python files, restart the kernel and run again.
Each build gets a new development tag and each execution its own output folder.
The named development pipeline is updated to the new image digest; CI-owned
pipelines are unchanged. For an isolated comparison, change `DEVELOPMENT`.

This notebook exercises managed training, including preparation, feature selection
and evaluation. It does not execute Batch Transform, approve a model, provision a
Feature Store or configure CI. The existing 03B/03C/03D notebooks remain available.

Clear all outputs before sharing this notebook: execution outputs contain your
environment's resource names and local paths.
